# ogbn-proteins: GNN Training with PyTorch Geometric (PyG)

This notebook trains a GNN on the **ogbn-proteins** benchmark from the Open Graph Benchmark (OGB)
using **PyTorch Geometric (PyG)**.

The model supports the following MPNN types:
- `sage` – GraphSAGE
- `gcn`  – Graph Convolutional Network
- `saint` 
- `SGCN` 



## 1. Install Dependencies

Run the cell below **once** if the required packages are not yet installed.

In [1]:
# Uncomment and run if packages are missing
# !pip install torch
# !pip install torch_geometric
# !pip install torch_scatter torch_sparse -f https://data.pyg.org/whl/torch-$(python -c 'import torch; print(torch.__version__)')+cu121.html
# !pip install ogb

## 2. Imports

In [2]:
import os

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.models import GNN_PyG
from src.utils import set_seed, load_data, preprocess, gen_model, add_labels, apply_ood_perturbation
from src.train import train_epoch, evaluate, run
from src.logging_utils import (
    setup_dirs, setup_experiment_dir, save_config, update_experiment_index,
    build_epoch_df, build_run_record,
    save_epoch_metrics, save_run_summary, save_aggregate_summary, compute_aggregate,
)
from src.visualization import (
    plot_loss_curve, plot_auc_curve, plot_auc_boxplot, plot_time_bar,
)

print('All imports successful.')


/root/miniconda3/envs/myconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All imports successful.


## 3. Configuration

Edit the variables below to configure the experiment (replaces command-line arguments).

In [3]:
# ── Device ──────────────────────────────────────────────────────────────────
USE_CPU  = False   # Set True to force CPU mode
GPU_ID   = 0       # GPU device ID (ignored when USE_CPU=True)

# ── Reproducibility ─────────────────────────────────────────────────────────
SEED     = 0
N_RUNS   = 1       # Number of independent runs

# ── Model ────────────────────────────────────────────────────────────────────
MPNN       = 'sgcn' # 'sage' | 'gcn' | 'graphsaint' | 'sgcn'
N_LAYERS   = 3
N_HIDDEN   = 64
USE_LABELS = False # Concatenate training labels as input features
JK         = False # Enable Jumping Knowledge (JK) aggregation

# ===== OOD Config =====
OOD_ENABLE = True

Pood = 0.01
# Bernoulli 抽样概率
# 表示多少比例节点成为 anomalous nodes
# 推荐后续测试:
# 0.01, 0.05, 0.10

Pcr = 0.20
# 对抽中的节点:
# 重连多少比例邻边
# 推荐:
# 0.10,0.20,0.30

OOD_RANDOM_SEED = 42

OOD_VERBOSE = True

# ── SGCN (only used when MPNN = 'sgcn') ────────────────────────────────────
N_SUBGRAPHS        = 32              # number of independent subgraphs per epoch
SUBSAMPLING_METHOD = 'random_node'  # 'random_node' | 'random_edge' | 'random_walk' | 'snowball'
SUBGRAPH_MAX_NODES = 5000           # hard upper bound on nodes per subgraph (<= 0: use SUBGRAPH_RATIO)
MAX_SUBGRAPH_EDGES = 0         # hard upper bound on edges per subgraph (<= 0: no cap)
TRUNCATION_RATIO   = 0.2            # fraction of worst subgraphs to discard (0.0 = keep all)
AGGREGATION_METHOD = 'sgcn'         # 'sgcn' (softmax) | 'avg' (SGCN-Avg) | 'weighted' (SGCN-Weighted)
LOCAL_EPOCHS       = 3              # number of local gradient steps per subgraph (L)
DEBUG_SUBGRAPH_STATS = False        # print per-subgraph shape/memory stats for OOM debugging
MIN_SUBGRAPH_NODES        = 0       # if > 0, pad subgraph to at least this many total nodes (0 = disabled)
MIN_TRAIN_NODES_IN_SUBGRAPH = 3000    # minimum training nodes guaranteed in each subgraph

# ── Regularisation ───────────────────────────────────────────────────────────
DROPOUT    = 0.25
INPUT_DROP = 0.1
EDGE_DROP  = 0.1

# ── Optimiser ────────────────────────────────────────────────────────────────
LR           = 0.01
WEIGHT_DECAY = 0.0

# ── Training schedule ────────────────────────────────────────────────────────
N_EPOCHS   = 200
EVAL_EVERY = 5
LOG_EVERY  = 5

# ── Misc ─────────────────────────────────────────────────────────────────────
SAVE_PRED  = False  # Save final predictions to results/<exp>/preds/

# ── Dataset constants (do not change) ────────────────────────────────────────
DATASET_NAME = 'ogbn-proteins'
N_NODE_FEATS = 0    # will be set after preprocessing
N_CLASSES    = 112

# ── Device setup ─────────────────────────────────────────────────────────────
if USE_CPU or not torch.cuda.is_available():
    device = torch.device('cpu')
else:
    device = torch.device(f'cuda:{GPU_ID}')

print(f'Using device: {device}')


Using device: cuda:0


## 4. Model Definition

GNN_PyG supports GraphSAGE and GCN.

In [4]:
# GNN_PyG is imported from src.models
print('Model class imported from src.models.')


Model class imported from src.models.


## 5. Utility Functions

In [5]:
# set_seed, load_data, preprocess, gen_model, add_labels, apply_ood_perturbation are imported from src.utils
print('Utility functions imported from src.utils.')


Utility functions imported from src.utils.


## 6. Training and Evaluation

In [6]:
# train_epoch, evaluate are imported from src.train
print('Training/evaluation functions imported from src.train.')


Training/evaluation functions imported from src.train.


## 7. Main Run Function

In [7]:
# run is imported from src.train
print('Run function imported from src.train.')


Run function imported from src.train.


## 8. Load and Preprocess Data

In [8]:
print('Loading data ...')
data, train_idx, val_idx, test_idx, evaluator = load_data(DATASET_NAME)

print('Preprocessing ...')
data = preprocess(data, train_idx, N_CLASSES)

ood_stats = None
clean_data = data
data_eval = clean_data
data_train = data
if data_train.edge_attr is not None and data_train.edge_attr.size(0) != data_train.edge_index.size(1):
    print("Drop mismatched train edge_attr:",
          data_train.edge_index.size(1), data_train.edge_attr.size(0))
    data_train.edge_attr = None

if data_eval.edge_attr is not None and data_eval.edge_attr.size(0) != data_eval.edge_index.size(1):
    print("Drop mismatched eval edge_attr:",
          data_eval.edge_index.size(1), data_eval.edge_attr.size(0))
    data_eval.edge_attr = None

if OOD_ENABLE:
    data_train, ood_stats = apply_ood_perturbation(
        clean_data,
        Pood=Pood,
        Pcr=Pcr,
        seed=OOD_RANDOM_SEED,
    )

    if OOD_VERBOSE:
        print(f'OOD enabled: {OOD_ENABLE}')
        print('Graph usage: train=OOD edge_index, eval=clean edge_index')
        print(f'Selected anomalous nodes: {ood_stats["selected_nodes"]}')
        print(f'Number: {ood_stats["num_selected_nodes"]}')
        print(f'Ratio: {ood_stats["selected_ratio"]:.6f}')
        print(f'Edges before/after: {ood_stats["num_edges_before"]} -> {ood_stats["num_edges_after"]}')
        print(f'Mean degree before: {ood_stats["mean_degree_before"]:.4f}')
        print(f'Mean degree after: {ood_stats["mean_degree_after"]:.4f}')
        print(f'Degree change summary: {ood_stats["degree_change_summary"]}')
        print(f'Total rewired edges: {ood_stats["rewired_edges"]}')
        print(f'Max rewired/node: {ood_stats["max_rewired_per_node"]}')
else:
    if OOD_VERBOSE:
        print(f'OOD enabled: {OOD_ENABLE}')
        print('Graph usage: train=clean edge_index, eval=clean edge_index')

data = clean_data
N_NODE_FEATS = data.x.shape[-1]

labels = clean_data.y
labels, train_idx, val_idx, test_idx = (
    labels.to(device),
    train_idx.to(device),
    val_idx.to(device),
    test_idx.to(device),
)

print(f'Node features:  {data.x.shape}')
print(f'Labels:         {labels.shape}')
print(f'Train nodes:    {len(train_idx)}')
print(f'Val nodes:      {len(val_idx)}')
print(f'Test nodes:     {len(test_idx)}')
print(f'N_NODE_FEATS (after preprocess): {N_NODE_FEATS}')


Loading data ...


/root/miniconda3/envs/myconda/lib/python3.10/site-packages/ogb/nodeproppred/dataset_pyg.py:69: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  self.data, self.slices = torch.l

Preprocessing ...
OOD enabled: True
Graph usage: train=OOD edge_index, eval=clean edge_index
Selected anomalous nodes: tensor([    29,     50,     74,  ..., 132290, 132315, 132522])
Number: 1336
Ratio: 0.010080
Edges before/after: 79122504 -> 79122502
Mean degree before: 596.3989
Mean degree after: 596.5007
Degree change summary: {'mean': 0.10179640352725983, 'std': 1.9167078733444214, 'max_abs': 10.0}
Total rewired edges: 158680
Max rewired/node: 708
Node features:  torch.Size([132534, 8])
Labels:         torch.Size([132534, 112])
Train nodes:    86619
Val nodes:      21236
Test nodes:     24679
N_NODE_FEATS (after preprocess): 8


## 9. Train the Model

In [9]:
config = {
    'method':          MPNN,
    'dataset':         DATASET_NAME,
    'sampling_method': SUBSAMPLING_METHOD,
    'trunc_ratio':     TRUNCATION_RATIO,
    'local_epochs':    LOCAL_EPOCHS,
    'epochs':          N_EPOCHS,
    'lr':              LR,
    'hidden_dim':      N_HIDDEN,
    'ood_enable':      OOD_ENABLE,
    'Pood':  Pood,
    'Pcr': Pcr,
    'ood_random_seed': OOD_RANDOM_SEED,
    'train_graph_mode': 'ood' if OOD_ENABLE else 'clean',
    'eval_graph_mode':  'clean',
    'seeds':           [SEED + i for i in range(N_RUNS)],
}

exp_dir, figures_dir = setup_experiment_dir('results')
pred_dir = os.path.join(exp_dir, 'preds')
os.makedirs(pred_dir, exist_ok=True)
save_config(config, exp_dir, device=device)

print(f'Experiment directory: {exp_dir}')
print(f'Prediction directory: {pred_dir}')

all_epoch_dfs  = []
all_run_records = []

for i in range(N_RUNS):
    print(f'\n=== Run {i + 1} / {N_RUNS} ===')
    seed = SEED + i
    set_seed(seed)
    _gen_model = lambda: gen_model(
        N_NODE_FEATS, N_CLASSES, USE_LABELS, N_LAYERS, N_HIDDEN,
        DROPOUT, INPUT_DROP, EDGE_DROP, MPNN, JK
    )
    result = run(
        data_eval, labels, train_idx, val_idx, test_idx, evaluator, n_running=i + 1,
        gen_model_fn=_gen_model, device=device,
        n_layers=N_LAYERS, lr=LR, weight_decay=WEIGHT_DECAY,
        n_epochs=N_EPOCHS, eval_every=EVAL_EVERY, log_every=LOG_EVERY,
        save_pred=SAVE_PRED, use_labels=USE_LABELS, n_classes=N_CLASSES,
        mpnn=MPNN,
        subsampling_method=SUBSAMPLING_METHOD,
        truncation_ratio=TRUNCATION_RATIO,
        aggregation_method=AGGREGATION_METHOD,
        n_subgraphs=N_SUBGRAPHS,
        subgraph_max_nodes=SUBGRAPH_MAX_NODES,
        max_subgraph_edges=MAX_SUBGRAPH_EDGES,
        local_epochs=LOCAL_EPOCHS,
        debug_subgraph_stats=DEBUG_SUBGRAPH_STATS,
        min_subgraph_nodes=MIN_SUBGRAPH_NODES,
        min_train_nodes_in_subgraph=MIN_TRAIN_NODES_IN_SUBGRAPH,
        data_train=data_train,
        data_eval=data_eval,
        pred_dir=pred_dir,
    )

    all_epoch_dfs.append(build_epoch_df(MPNN, i + 1, seed, result['epoch_records']))
    all_run_records.append(build_run_record(MPNN, i + 1, seed, result))

    print(
        f'Run {i + 1} finished – '
        f'Val ROC-AUC: {result["best_val_auc"]:.4f} | '
        f'Test ROC-AUC: {result["best_test_auc"]:.4f} | '
        f'Total time: {result["total_run_time"]:.1f}s'
    )

all_val_scores  = [r['best_val_auc']  for r in all_run_records]
all_test_scores = [r['best_test_auc'] for r in all_run_records]
print('\n=== Summary ===')
print(f'Val  ROC-AUC: {np.mean(all_val_scores):.4f} ± {np.std(all_val_scores):.4f}')
print(f'Test ROC-AUC: {np.mean(all_test_scores):.4f} ± {np.std(all_test_scores):.4f}')



=== Run 1 / 1 ===


/root/miniconda3/envs/myconda/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:60: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch: 0005 | Loss: 0.3123 | Train: 72.13% | Valid: 66.26% | Test: 59.49% | Best Valid: 66.26% | Best Test: 59.49%
Epoch: 0010 | Loss: 0.2975 | Train: 75.47% | Valid: 69.54% | Test: 60.05% | Best Valid: 69.54% | Best Test: 60.05%
Epoch: 0015 | Loss: 0.2925 | Train: 76.80% | Valid: 71.62% | Test: 62.14% | Best Valid: 71.62% | Best Test: 62.14%
Epoch: 0020 | Loss: 0.2905 | Train: 77.77% | Valid: 72.84% | Test: 62.97% | Best Valid: 72.84% | Best Test: 62.97%
Epoch: 0025 | Loss: 0.2876 | Train: 78.33% | Valid: 73.38% | Test: 63.50% | Best Valid: 73.38% | Best Test: 63.50%
Epoch: 0030 | Loss: 0.2865 | Train: 78.88% | Valid: 74.10% | Test: 64.08% | Best Valid: 74.10% | Best Test: 64.08%
Epoch: 0035 | Loss: 0.2851 | Train: 78.56% | Valid: 73.13% | Test: 64.07% | Best Valid: 74.10% | Best Test: 64.08%
Epoch: 0040 | Loss: 0.2833 | Train: 78.65% | Valid: 73.65% | Test: 64.21% | Best Valid: 74.10% | Best Test: 64.08%
Epoch: 0045 | Loss: 0.2819 | Train: 79.59% | Valid: 74.68% | Test: 65.09% | Best

## 10. Save Results to CSV


In [10]:
epoch_df = save_epoch_metrics(all_epoch_dfs, exp_dir)
run_df   = save_run_summary(all_run_records, exp_dir)
agg_df   = save_aggregate_summary(all_run_records, exp_dir)

update_experiment_index(exp_dir, config, compute_aggregate(all_run_records))

print('\naggregate_summary:')
print(agg_df.to_string(index=False))


Saved: results/exp_20260518_002444/config.yaml
Saved: results/exp_20260518_002444/epoch_metrics.csv
Saved: results/exp_20260518_002444/run_summary.csv
Saved: results/exp_20260518_002444/aggregate_summary.csv
Updated: results/experiment_index.csv

aggregate_summary:
method  n_runs  mean_best_val_auc  std_best_val_auc  mean_best_test_auc  std_best_test_auc  mean_train_epoch_time  std_train_epoch_time  mean_eval_time  std_eval_time  mean_total_run_time  std_total_run_time  local_epochs  mean_sgcn_epoch_time_max  std_sgcn_epoch_time_max  mean_max_subgraph_pipeline_time  std_max_subgraph_pipeline_time  mean_aggregation_time  std_aggregation_time
  sgcn       1           0.774005               NaN            0.706065                NaN                 0.0565                   NaN        2.245423            NaN           642.282059                 NaN             3                    0.0565                      NaN                         0.053796                             NaN              

## 11. Generate Figures


In [11]:
plot_loss_curve(epoch_df, figures_dir, method=MPNN)
plot_auc_curve(epoch_df, figures_dir, method=MPNN)
plot_auc_boxplot(run_df, figures_dir, method=MPNN)
plot_time_bar(compute_aggregate(all_run_records), figures_dir, method=MPNN)

print('All figures saved to', figures_dir)


Saved: results/exp_20260518_002444/figures/loss_curve.png
Saved: results/exp_20260518_002444/figures/auc_curve.png
Saved: results/exp_20260518_002444/figures/auc_boxplot.png
Saved: results/exp_20260518_002444/figures/time_bar.png
All figures saved to results/exp_20260518_002444/figures
